# 03 — Model Tuning (Step 5)
Ít nhất 5 kỹ thuật:
1. GridSearchCV — Random Forest
2. GridSearchCV — SVM
3. GridSearchCV — kNN
4. Class weight balancing (xử lý imbalanced: 150/35/30)
5. Stacking Ensemble (RF + SVM + kNN → Logistic Regression)

So sánh baseline vs tuned sau cùng.

In [1]:
import sys; sys.path.append('../src')
from pathlib import Path
import pandas as pd, warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from utils import save_model, plot_roc_curves, plot_confusion_matrix
from evaluate import evaluate_model

SEED = 42
TUNED_DIR = Path('../results/tuned')
FIG_DIR = Path('../results/figures')
MODEL_DIR = Path('../models/tuned')
for p in [TUNED_DIR, FIG_DIR / 'roc_curves', FIG_DIR / 'confusion_matrix', MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../data/processed/thyroid_processed.csv')
X = df.drop('class', axis=1); y = df['class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


## Kỹ thuật 1: GridSearchCV — Random Forest

In [2]:
param_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=SEED), param_rf, cv=cv, scoring='f1_weighted', n_jobs=1)
grid_rf.fit(X_train, y_train)
print('Best params RF :', grid_rf.best_params_)
print('Best CV F1     :', round(grid_rf.best_score_, 4))


Best params RF : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 50}
Best CV F1     : 0.9409


## Kỹ thuật 2: GridSearchCV — SVM

In [3]:
param_svm = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__kernel': ['rbf', 'linear'],
    'svc__gamma': ['scale', 'auto']
}
svm_pipe = make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED))
grid_svm = GridSearchCV(svm_pipe, param_svm, cv=cv, scoring='f1_weighted', n_jobs=1)
grid_svm.fit(X_train, y_train)
print('Best params SVM:', grid_svm.best_params_)
print('Best CV F1     :', round(grid_svm.best_score_, 4))


Best params SVM: {'svc__C': 100, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
Best CV F1     : 0.9666


## Kỹ thuật 3: GridSearchCV — kNN

In [4]:
param_knn = {
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 9, 11, 15],
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__metric': ['euclidean', 'manhattan']
}
knn_pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid_knn = GridSearchCV(knn_pipe, param_knn, cv=cv, scoring='f1_weighted', n_jobs=1)
grid_knn.fit(X_train, y_train)
print('Best params kNN:', grid_knn.best_params_)
print('Best CV F1     :', round(grid_knn.best_score_, 4))


Best params kNN: {'kneighborsclassifier__metric': 'euclidean', 'kneighborsclassifier__n_neighbors': 3, 'kneighborsclassifier__weights': 'distance'}
Best CV F1     : 0.9379


## Kỹ thuật 4: Class weight balancing
Dataset imbalanced: Normal=150, Hyper=35, Hypo=30 → class_weight='balanced'

In [5]:
best_rf_params = {k: v for k, v in grid_rf.best_params_.items()}
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=SEED, **best_rf_params)
rf_balanced.fit(X_train, y_train)

m_bal = evaluate_model(rf_balanced, X_test, y_test)
m_tun = evaluate_model(grid_rf.best_estimator_, X_test, y_test)

print('RF tuned    :', m_tun)
print('RF balanced :', m_bal)


RF tuned    : {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'AUC': 1.0}
RF balanced : {'Accuracy': 0.97, 'Precision': 0.97, 'Recall': 0.97, 'F1-Score': 0.97, 'AUC': 1.0}


## Kỹ thuật 5: Stacking Ensemble

In [6]:
estimators = [
    ('rf', grid_rf.best_estimator_),
    ('svm', grid_svm.best_estimator_),
    ('knn', grid_knn.best_estimator_),
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=5, passthrough=False, n_jobs=1
)
stack.fit(X_train, y_train)
m_stack = evaluate_model(stack, X_test, y_test)
print('Stacking:', m_stack)


Stacking: {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'AUC': 1.0}


## Tổng hợp: Baseline vs Tuned

In [7]:
baseline = pd.read_csv('../results/baseline/metrics_table.csv')
best_baseline = baseline.loc[baseline['F1-Score'].idxmax()]

tuned_rows = [
    {'Model': 'RF (GridSearch)', **evaluate_model(grid_rf.best_estimator_, X_test, y_test)},
    {'Model': 'SVM (GridSearch)', **evaluate_model(grid_svm.best_estimator_, X_test, y_test)},
    {'Model': 'kNN (GridSearch)', **evaluate_model(grid_knn.best_estimator_, X_test, y_test)},
    {'Model': 'RF + balanced', **evaluate_model(rf_balanced, X_test, y_test)},
    {'Model': 'Stacking', **m_stack},
]
df_tuned = pd.DataFrame(tuned_rows)
df_tuned.to_csv(TUNED_DIR / 'metrics_table.csv', index=False)

print(f'Best baseline ({best_baseline["Model"]}): F1={best_baseline["F1-Score"]}')
print('\nTuned results:')
print(df_tuned[['Model','F1-Score','AUC','Accuracy']].to_string(index=False))

best_tuned_idx = df_tuned['F1-Score'].idxmax()
best_name = df_tuned.loc[best_tuned_idx, 'Model']
model_map = {
    'RF (GridSearch)': grid_rf.best_estimator_,
    'SVM (GridSearch)': grid_svm.best_estimator_,
    'kNN (GridSearch)': grid_knn.best_estimator_,
    'RF + balanced': rf_balanced,
    'Stacking': stack,
}
save_model(model_map[best_name], 'best_model', MODEL_DIR)
print(f'\nBest tuned model saved: {best_name}')


Best baseline (NaiveBayes): F1=0.98

Tuned results:
           Model  F1-Score  AUC  Accuracy
 RF (GridSearch)      1.00  1.0      1.00
SVM (GridSearch)      0.97  1.0      0.97
kNN (GridSearch)      0.97  1.0      0.97
   RF + balanced      0.97  1.0      0.97
        Stacking      1.00  1.0      1.00
Saved → ..\models\tuned\best_model.pkl

Best tuned model saved: RF (GridSearch)
